In [1]:
!pip install decord

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 28.3 MB/s eta 0:00:00


In [2]:
import json
import numpy as np
import torch
import torchvision.transforms as transforms
from torchvision.models import resnet18
from decord import VideoReader, cpu
from PIL import Image
from pathlib import Path
from tqdm import tqdm
import math
import logging
import re
import warnings
from sklearn.metrics.pairwise import cosine_similarity
from collections import defaultdict
from typing import List, Dict, Optional
from IPython.display import Video, display
import subprocess
import os
from google.colab import drive
from google.colab import files

import random

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.WARNING)

## Technical Implementation

**Visual Feature Extraction:**
- ResNet18 (pretrained on ImageNet) extracts 512-dimensional features
- 8 frames sampled uniformly per 30-second segment
- Features averaged and L2-normalized for retrieval

**Text Processing:**
- Transcript parsing with regex for `[HH:MM:SS - HH:MM:SS] text` format
- Temporal alignment between transcript segments and video chunks
- Phase classification using keyword matching against predefined vocabularies

**Retrieval System:**
- Dual-modal search: keyword matching in transcripts + cosine similarity on visual features
- Hybrid ranking combining text relevance scores and visual similarity
- Returns most relevant segments with synchronized transcripts and video clips

**Dataset Creation:**
- Processes 4 surgical videos into 77 training examples (30-second segments each)
- Each example contains: video_features (512-dim), transcript, surgical_phase, timestamps, video_path
- Saves as `surgirag_data.json` - structured dataset ready for RAG training/evaluation
- Compatible with standard RAG frameworks for model fine-tuning

## Three-Cell Structure

**Cell 1: Setup & Processing**
- Mounts Google Drive and loads 4 cholecystectomy videos (.mp4)
- Segments videos into 30-second chunks (77 total segments)
- Extracts ResNet18 features from 8 uniformly sampled frames per segment
- Parses transcript files and aligns with video timestamps
- Classifies surgical phases (preparation, exploration, dissection, clipping, extraction)
- Saves processed dataset to `surgirag_data.json`

**Cell 2: Validation & Testing**
- Tests data quality: transcript coverage (98.7%), phase distribution, feature dimensions
- Validates transcript-phase alignment using keyword matching
- Tests retrieval accuracy: same-phase retrieval, temporal coherence, keyword precision
- Verifies the processed dataset is ready for RAG training

**Cell 3: Video Retrieval & Question Answering**
- Implements search functions: keyword-based, phase-based, and semantic similarity
- Extracts relevant video clips using FFmpeg with proper encoding
- Main interface: `ask_question()` function for natural language queries
- Downloads video clips automatically for local playback

## Usage Examples

```python
ask_question(surgirag, "How do you dissect the cystic artery?")
ask_question(surgirag, "Show me trocar insertion")
ask_question(surgirag, "What happens during gallbladder removal?")
ask_question(surgirag, "How do you achieve critical view of safety?")
```

The system retrieves the most relevant 30-second video segment with synchronized transcript and downloads the clip.

In [6]:
# Setup SurgiRAG System

class SurgiRAG:
    """Main SurgiRAG system for processing surgical videos and retrieving relevant segments"""

    def __init__(self,
                 videos_dir="/content/drive/MyDrive/DATA266/final_project/surgical_videos",
                 transcripts_dir="/content/drive/MyDrive/DATA266/final_project/transcripts",
                 output_dir="./surgirag_data",
                 seg_seconds=30,
                 frames_per_seg=8):

        self.videos_dir = Path(videos_dir)
        self.transcripts_dir = Path(transcripts_dir)
        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(exist_ok=True)

        self.seg_seconds = seg_seconds
        self.frames_per_seg = frames_per_seg
        self.device = "cuda" if torch.cuda.is_available() else "cpu"

        self.data = []
        self.video_embeddings = None

        self._setup_model()
        self._setup_phases()

    def _setup_model(self):
        """Initialize ResNet model for visual feature extraction"""
        self.visual_model = resnet18(pretrained=True)
        self.visual_model = torch.nn.Sequential(*list(self.visual_model.children())[:-1])

        if self.device == "cuda":
            self.visual_model = self.visual_model.cuda()
        self.visual_model.eval()

        self.transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

    def _setup_phases(self):
        """Define surgical phases with their descriptions and keywords"""
        self.phases = {
            "preparation": {
                "description": "Patient positioning, trocar insertion, pneumoperitoneum creation",
                "keywords": ["port", "trocar", "insufflation", "pneumoperitoneum", "setup", "prep", "incision", "umbilical"]
            },
            "exploration": {
                "description": "Initial laparoscopic exploration and anatomy identification",
                "keywords": ["explore", "look", "see", "identify", "anatomy", "gallbladder", "liver", "examine", "fundus"]
            },
            "dissection": {
                "description": "Dissection and critical view achievement",
                "keywords": ["dissect", "calot", "triangle", "artery", "duct", "critical view", "safety", "clear", "cystic"]
            },
            "clipping": {
                "description": "Clipping and division of structures",
                "keywords": ["clip", "clipping", "divide", "cut", "scissors", "secure", "tie", "knot"]
            },
            "extraction": {
                "description": "Gallbladder extraction and closure",
                "keywords": ["extract", "bag", "remove", "gallbladder", "close", "closure", "specimen"]
            }
        }

    def parse_transcript(self, transcript_path):
        """Parse transcript file with timestamps into segments"""
        try:
            with open(transcript_path, 'r', encoding='utf-8') as f:
                content = f.read()

            segments = []
            # Parse format: [00:00:09 - 00:00:10] text
            pattern = r'\[(\d{2}):(\d{2}):(\d{2}) - (\d{2}):(\d{2}):(\d{2})\]\s*(.+?)(?=\[|\Z)'
            matches = re.findall(pattern, content, re.DOTALL)

            for match in matches:
                start_h, start_m, start_s, end_h, end_m, end_s, text = match
                start_time = int(start_h) * 3600 + int(start_m) * 60 + int(start_s)
                end_time = int(end_h) * 3600 + int(end_m) * 60 + int(end_s)

                clean_text = text.strip().replace('\n', ' ')
                if clean_text:
                    segments.append({
                        'start': start_time,
                        'end': end_time,
                        'text': clean_text
                    })

            return segments
        except:
            return []

    def get_transcript_for_segment(self, transcript_segments, start_time, end_time):
        """Get transcript text that overlaps with video segment time range"""
        if not transcript_segments:
            return ""

        segment_texts = []
        for seg in transcript_segments:
            # check if transcript segment overlaps with video segment
            if seg['start'] < end_time and seg['end'] > start_time:
                segment_texts.append(seg['text'])

        return " ".join(segment_texts).strip()

    def classify_phase(self, transcript_text):
        """Classify surgical phase based on keywords in transcript"""
        if not transcript_text or len(transcript_text.strip()) < 3:
            return "unknown"

        text_lower = transcript_text.lower()
        phase_scores = {}

        for phase_name, phase_info in self.phases.items():
            score = 0
            for keyword in phase_info["keywords"]:
                if keyword.lower() in text_lower:
                    score += len(keyword.split()) * 2
                    # for specific medical terms
                    if keyword in ["critical view", "cystic artery", "cystic duct"]:
                        score += 3
            phase_scores[phase_name] = score

        if max(phase_scores.values()) > 0:
            return max(phase_scores, key=phase_scores.get)

        return "unknown"

    def extract_visual_features(self, video_path, start_s, end_s):
        """Extract visual features from video segment using ResNet"""
        try:
            vr = VideoReader(str(video_path), ctx=cpu(0))
            fps = vr.get_avg_fps()
            start_f = int(start_s * fps)
            end_f = int(end_s * fps)
            max_frame = len(vr) - 1

            end_f = min(end_f, max_frame)
            start_f = min(start_f, max_frame)

            if start_f >= end_f:
                return torch.nn.functional.normalize(torch.randn(512), dim=-1)

            # sample frames uniformly across the segment
            frame_indices = np.linspace(start_f, end_f, self.frames_per_seg, dtype=int)
            frame_features = []

            for idx in frame_indices:
                try:
                    frame = vr[int(idx)]
                    if hasattr(frame, "asnumpy"):
                        arr = frame.asnumpy()
                    else:
                        arr = frame.cpu().numpy()

                    if arr.dtype != np.uint8:
                        arr = (arr * 255).astype(np.uint8)

                    pil_frame = Image.fromarray(arr)
                    frame_tensor = self.transform(pil_frame).unsqueeze(0)

                    if self.device == "cuda":
                        frame_tensor = frame_tensor.cuda()

                    with torch.no_grad():
                        features = self.visual_model(frame_tensor).squeeze().cpu()
                        frame_features.append(features)

                except:
                    continue

            # avg features across frames
            if frame_features:
                avg_features = torch.stack(frame_features).mean(dim=0)
                return torch.nn.functional.normalize(avg_features, dim=-1)
            else:
                return torch.nn.functional.normalize(torch.randn(512), dim=-1)

        except:
            return torch.nn.functional.normalize(torch.randn(512), dim=-1)

    def find_transcript_file(self, video_path):
        """Find matching transcript file for a video"""
        video_stem = video_path.stem

        # different naming patterns
        possible_names = [
            f"{video_stem}_transcript.txt",
            f"{video_stem.replace(' ', '_')}_transcript.txt",
            f"{video_stem.replace('：', '_')}_transcript.txt",
        ]

        for name in possible_names:
            transcript_path = self.transcripts_dir / name
            if transcript_path.exists():
                return transcript_path

        # Try partial matching
        for transcript_file in self.transcripts_dir.glob("*.txt"):
            if video_stem.split()[0] in transcript_file.stem:
                return transcript_file

        return None

    def process_video(self, video_path):
        """Process single video into segments with features and transcripts"""
        video_id = video_path.stem

        # find and parse transcript
        transcript_path = self.find_transcript_file(video_path)
        if transcript_path:
            transcript_segments = self.parse_transcript(transcript_path)
        else:
            transcript_segments = []

        # get video metadata
        try:
            vr = VideoReader(str(video_path), ctx=cpu(0))
            fps = vr.get_avg_fps()
            total_frames = len(vr)
            total_seconds = total_frames / fps
            n_segments = math.ceil(total_seconds / self.seg_seconds)
            del vr
        except:
            return []

        video_segments = []

        for seg_idx in tqdm(range(n_segments), desc=f"Processing {video_id}"):
            try:
                start_s = seg_idx * self.seg_seconds
                end_s = min((seg_idx + 1) * self.seg_seconds, total_seconds)

                # get transcript for this time segment
                segment_transcript = self.get_transcript_for_segment(transcript_segments, start_s, end_s)

                # classify surgical phase
                if segment_transcript:
                    phase = self.classify_phase(segment_transcript)
                else:
                    # fallback (classify based on video progress)
                    progress = start_s / total_seconds
                    if progress < 0.2:
                        phase = "preparation"
                    elif progress < 0.4:
                        phase = "exploration"
                    elif progress < 0.7:
                        phase = "dissection"
                    elif progress < 0.9:
                        phase = "clipping"
                    else:
                        phase = "extraction"

                # visual features
                features = self.extract_visual_features(video_path, start_s, end_s)

                # segment data
                segment = {
                    'video_id': video_id,
                    'video_path': str(video_path),
                    'segment_idx': seg_idx,
                    'start_time': start_s,
                    'end_time': end_s,
                    'duration': end_s - start_s,
                    'phase': phase,
                    'phase_description': self.phases.get(phase, {}).get('description', ''),
                    'transcript': segment_transcript,
                    'has_transcript': bool(segment_transcript and segment_transcript.strip()),
                    'features': features.tolist(),
                    'confidence': 0.9 if segment_transcript else 0.6
                }

                video_segments.append(segment)

            except:
                continue

        return video_segments

    def process_all_videos(self):
        """Process all videos in the directory"""
        video_files = []
        for ext in ['.mp4', '.avi', '.mov', '.mkv']:
            video_files.extend(self.videos_dir.glob(f"*{ext}"))

        if not video_files:
            print(f"No video files found in {self.videos_dir}")
            return

        all_segments = []
        for video_file in video_files:
            try:
                segments = self.process_video(video_file)
                all_segments.extend(segments)
            except Exception as e:
                print(f"Failed to process {video_file.name}: {e}")

        self.data = all_segments
        self.video_embeddings = np.array([seg['features'] for seg in self.data])

        # save processed data
        self.save_data()

    def save_data(self):
        """Save processed segments to JSON file"""
        if not self.data:
            return

        output_file = self.output_dir / "surgirag_data.json"
        with open(output_file, 'w') as f:
            json.dump(self.data, f, indent=2)

        with_transcript = sum(1 for seg in self.data if seg['has_transcript'])
        phases = {}
        for seg in self.data:
            phase = seg['phase']
            phases[phase] = phases.get(phase, 0) + 1

        print(f"Processed {len(self.data)} segments ({with_transcript} with transcripts)")
        print(f"Phase distribution: {phases}")

    def load_data(self, data_path=None):
        """Load previously processed data"""
        if data_path is None:
            data_path = self.output_dir / "surgirag_data.json"

        try:
            with open(data_path, 'r') as f:
                self.data = json.load(f)

            self.video_embeddings = np.array([seg['features'] for seg in self.data])
            print(f"Loaded {len(self.data)} segments")

        except Exception as e:
            print(f"Failed to load data: {e}")

# setup function
def setup_surgirag():
    """Setup SurgiRAG system with drive mounting and data loading"""
    # mount google drive
    try:
        drive.mount('/content/drive')
    except:
        print("Failed to mount Google Drive")
        return None

    # Initialize SurgiRAG
    surgirag = SurgiRAG()

    # Check if processed data already exists
    data_file = surgirag.output_dir / "surgirag_data.json"
    if data_file.exists():
        surgirag.load_data()
    else:
        print("Processing videos...")
        surgirag.process_all_videos()

    return surgirag

# Run setup
surgirag = setup_surgirag()
print("SurgiRAG setup complete")

Mounted at /content/drive


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100%|██████████| 44.7M/44.7M [00:00<00:00, 135MB/s]


Processing videos...


Processing Cholecystectomy by Cal Shipley, M.D.: 100%|██████████| 8/8 [00:18<00:00,  2.30s/it]
Processing Laparoscopic cholecystectomy for Mirizzi syndrome: 100%|██████████| 13/13 [00:44<00:00,  3.39s/it]
Processing Laparoscopic Cholecystectomy Full HD Video: 100%|██████████| 23/23 [01:12<00:00,  3.15s/it]
Processing Laparoscopic Cholecystectomy： Step- by- Step Surgical guide for Patients and Professionals＂: 100%|██████████| 33/33 [01:45<00:00,  3.19s/it]

Processed 77 segments (76 with transcripts)
Phase distribution: {'preparation': 9, 'exploration': 19, 'dissection': 31, 'extraction': 7, 'clipping': 7, 'unknown': 4}
SurgiRAG setup complete


In [7]:
# Validation and Testing

def validate_data_quality(surgirag):
    """Test data quality and transcript alignment"""
    if not surgirag.data:
        print("No data to validate")
        return

    # Basic data stats
    total_segments = len(surgirag.data)
    with_transcript = sum(1 for seg in surgirag.data if seg['has_transcript'])

    print(f"Total segments: {total_segments}")
    print(f"With transcripts: {with_transcript} ({with_transcript/total_segments*100:.1f}%)")

    # Phase distribution
    phases = {}
    for seg in surgirag.data:
        phase = seg['phase']
        phases[phase] = phases.get(phase, 0) + 1

    print(f"Phase distribution: {phases}")

    # Feature dimensions
    feature_lens = [len(seg['features']) for seg in surgirag.data[:5]]
    print(f"Feature dimensions: {set(feature_lens)}")

def test_transcript_alignment(surgirag):
    """Test if transcripts align with surgical phases"""
    print("\nTranscript-Phase Alignment")

    # Test if phase keywords appear in transcripts
    phase_keywords = {
        'preparation': ['port', 'trocar', 'insufflation'],
        'exploration': ['gallbladder', 'liver', 'fundus'],
        'dissection': ['calot', 'triangle', 'artery', 'duct'],
        'clipping': ['clip', 'tie', 'knot'],
        'extraction': ['remove', 'specimen', 'bag']}

    alignment_scores = {}
    for phase in phase_keywords.keys():
        phase_segments = [seg for seg in surgirag.data if seg['phase'] == phase and seg['has_transcript']]

        if not phase_segments:
            continue

        keywords = phase_keywords[phase]
        matches = 0

        for seg in phase_segments[:5]:  # Test first 5 segments
            transcript = seg['transcript'].lower()
            if any(keyword in transcript for keyword in keywords):
                matches += 1

        if phase_segments:
            alignment_scores[phase] = matches / min(len(phase_segments), 5)

    for phase, score in alignment_scores.items():
        print(f"{phase}: {score:.2f} alignment score")

def test_retrieval_quality(surgirag):
    """Test retrieval system quality"""
    # Test same-phase retrieval
    phase_accuracy = {}

    for phase in ['preparation', 'dissection', 'extraction']:
        phase_segments = [i for i, seg in enumerate(surgirag.data) if seg['phase'] == phase]

        if len(phase_segments) < 2:
            continue

        # Test with random segment from this phase
        query_idx = random.choice(phase_segments)
        query_embedding = surgirag.video_embeddings[query_idx:query_idx+1]

        # Find top-5 similar segments
        similarities = cosine_similarity(query_embedding, surgirag.video_embeddings)[0]
        top_indices = np.argsort(similarities)[::-1][1:6]  # Exclude self

        # Check how many are same phase
        same_phase_count = sum(1 for idx in top_indices if surgirag.data[idx]['phase'] == phase)
        phase_accuracy[phase] = same_phase_count / 5

    for phase, accuracy in phase_accuracy.items():
        print(f"{phase}: {accuracy:.2f} same-phase retrieval accuracy")

def test_keyword_retrieval(surgirag):
    """Test keyword-based retrieval"""

    test_keywords = {
        'cystic artery': 'dissection',
        'trocar': 'preparation',
        'gallbladder': 'exploration',
        'clip': 'clipping',
        'remove': 'extraction'
    }

    for keyword, expected_phase in test_keywords.items():
        # find segments containing keyword
        keyword_segments = []
        for i, seg in enumerate(surgirag.data):
            if keyword.lower() in seg['transcript'].lower():
                keyword_segments.append((i, seg))

        if not keyword_segments:
            print(f"{keyword}: No segments found")
            continue

        # check if found segments are mostly from expected phase
        phase_matches = sum(1 for _, seg in keyword_segments if seg['phase'] == expected_phase)
        accuracy = phase_matches / len(keyword_segments) if keyword_segments else 0

        print(f"{keyword}: {len(keyword_segments)} segments found, {accuracy:.2f} phase accuracy")

def test_temporal_coherence(surgirag):
    """Test if segments are temporally coherent"""

    # group by video
    videos = {}
    for seg in surgirag.data:
        video = seg['video_path']
        if video not in videos:
            videos[video] = []
        videos[video].append(seg)

    for video_path, segments in videos.items():
        video_name = Path(video_path).name

        # check timestamp order
        timestamps = [(seg['start_time'], seg['end_time']) for seg in segments]
        timestamps.sort()

        # check for overlaps or large gaps
        overlaps = 0
        gaps = 0

        for i in range(len(timestamps) - 1):
            current_end = timestamps[i][1]
            next_start = timestamps[i + 1][0]

            if current_end > next_start:
                overlaps += 1
            elif next_start - current_end > 5:  # Gap > 5 seconds
                gaps += 1

        print(f"{video_name}: {len(segments)} segments, {overlaps} overlaps, {gaps} gaps")

def sample_segments_by_phase(surgirag, phase, n=3):
    """Show sample segments from a specific phase"""
    phase_segments = [seg for seg in surgirag.data if seg['phase'] == phase and seg['has_transcript']]

    if not phase_segments:
        print(f"No segments found for phase: {phase}")
        return

    print(f"\nSample {phase} segments:")
    sample_segments = random.sample(phase_segments, min(n, len(phase_segments)))

    for i, seg in enumerate(sample_segments, 1):
        video_name = Path(seg['video_path']).name
        transcript = seg['transcript'][:100] + "..." if len(seg['transcript']) > 100 else seg['transcript']

        print(f"{i}. {video_name} ({seg['start_time']:.1f}s-{seg['end_time']:.1f}s)")
        print(f"   Transcript: \"{transcript}\"")

def run_all_validation_tests(surgirag):
    """Run comprehensive validation"""
    print("SurgiRAG Validation Suite")
    validate_data_quality(surgirag)
    test_transcript_alignment(surgirag)
    test_retrieval_quality(surgirag)
    test_keyword_retrieval(surgirag)
    test_temporal_coherence(surgirag)

    print("\nSample segments by phase:")
    for phase in ['preparation', 'dissection', 'extraction']:
        sample_segments_by_phase(surgirag, phase, n=2)

    # assessment
    transcript_coverage = sum(1 for seg in surgirag.data if seg['has_transcript']) / len(surgirag.data)

    if transcript_coverage > 0.8:
        print("Status: Ready for retrieval and question answering")
    elif transcript_coverage > 0.5:
        print("Status: Good coverage, may need minor improvements")
    else:
        print("Status: Low transcript coverage, needs improvement")

# run validation tests
run_all_validation_tests(surgirag)

SurgiRAG Validation Suite
Total segments: 77
With transcripts: 76 (98.7%)
Phase distribution: {'preparation': 9, 'exploration': 19, 'dissection': 31, 'extraction': 7, 'clipping': 7, 'unknown': 4}
Feature dimensions: {512}

Transcript-Phase Alignment
preparation: 0.80 alignment score
exploration: 0.80 alignment score
dissection: 1.00 alignment score
clipping: 0.80 alignment score
extraction: 1.00 alignment score
preparation: 0.40 same-phase retrieval accuracy
dissection: 0.60 same-phase retrieval accuracy
extraction: 0.00 same-phase retrieval accuracy
cystic artery: 15 segments found, 0.93 phase accuracy
trocar: 2 segments found, 0.50 phase accuracy
gallbladder: 37 segments found, 0.38 phase accuracy
clip: 12 segments found, 0.25 phase accuracy
remove: 11 segments found, 0.55 phase accuracy
Cholecystectomy by Cal Shipley, M.D..mp4: 8 segments, 0 overlaps, 0 gaps
Laparoscopic cholecystectomy for Mirizzi syndrome.mp4: 13 segments, 0 overlaps, 0 gaps
Laparoscopic Cholecystectomy Full HD Vi

In [8]:
# Video Retrieval and Playback

def search_segments(surgirag, query, top_k=5, method='auto'):
    """Search for relevant video segments"""
    if not surgirag.data:
        print("No data loaded")
        return []

    query_lower = query.lower()
    results = []

    # keyword search for specific medical terms
    if "cystic artery" in query_lower:
        # cystic artery segments in dissection phase
        for seg in surgirag.data:
            if "cystic artery" in seg['transcript'].lower() and seg['phase'] == 'dissection':
                results.append(seg)
    elif "trocar" in query_lower:
        # trocar segments
        for seg in surgirag.data:
            if "trocar" in seg['transcript'].lower():
                results.append(seg)
    elif any(phase in query_lower for phase in surgirag.phases.keys()):
        # by surgical phase
        for phase in surgirag.phases.keys():
            if phase in query_lower:
                results = [seg for seg in surgirag.data if seg['phase'] == phase]
                break
    else:
        # keyword search
        keywords = query_lower.split()
        for seg in surgirag.data:
            transcript = seg.get('transcript', '').lower()
            if any(word in transcript for word in keywords):
                # Calculate relevance score based on keyword frequency
                score = sum(transcript.count(word) for word in keywords)
                seg_copy = seg.copy()
                seg_copy['relevance_score'] = score
                results.append(seg_copy)

        # sort by relevance
        results.sort(key=lambda x: x.get('relevance_score', 0), reverse=True)

    # sort results chronologically within same video
    results.sort(key=lambda x: (x['video_path'], x['start_time']))

    return results[:top_k]

def extract_video_clip(segment, output_path=None):
    """Extract video clip for a specific segment using ffmpeg"""
    if output_path is None:
        output_path = f"clip_{segment['video_id']}_{segment['segment_idx']}.mp4"

    video_path = segment['video_path']
    start_time = segment['start_time']
    end_time = segment['end_time']

    # use ffmpeg to extract clip with better encoding
    cmd = [
        'ffmpeg', '-i', video_path,
        '-ss', str(start_time),
        '-t', str(end_time - start_time),
        '-c:v', 'libx264', '-preset', 'fast', '-crf', '28',
        '-c:a', 'aac',
        '-y', output_path]

    try:
        result = subprocess.run(cmd, check=True, capture_output=True)
        return output_path
    except:
        return None

def display_video_segment(segment):
    """Display video segment with information in Colab"""
    print(f"Video: {Path(segment['video_path']).name}")
    print(f"Time: {segment['start_time']:.1f}s - {segment['end_time']:.1f}s ({segment['duration']:.1f}s)")
    print(f"Phase: {segment['phase']} - {segment['phase_description']}")

    if segment['has_transcript']:
        print(f"Transcript: \"{segment['transcript']}\"")
    else:
        print("Transcript: Not available")

    print(f"Confidence: {segment['confidence']:.2f}")

    # extract video clip
    clip_path = extract_video_clip(segment)
    if clip_path and os.path.exists(clip_path):
        print(f"\nVideo clip created: {clip_path}")
        print(f"File size: {os.path.getsize(clip_path)} bytes")

        print(f"Downloading video clip")
        files.download(clip_path)

        return clip_path
    else:
        print("Failed to extract video clip")
        return None

def ask_question(surgirag, question, show_alternatives=True):
    """Ask a question and get video segment with playback"""
    print(f"Question: {question}")
    print("Searching for relevant video segments...")

    # search for relevant segments
    results = search_segments(surgirag, question, top_k=5)

    if not results:
        print("No relevant segments found.")
        return

    # display best match
    best_segment = results[0]
    print(f"\nFound {len(results)} relevant segments. Showing best match:")

    clip_path = display_video_segment(best_segment)

    # show alternatives if requested
    if show_alternatives and len(results) > 1:
        print(f"\nAlternative segments:")
        for i, segment in enumerate(results[1:], 2):
            print(f"\n{i}. {Path(segment['video_path']).name}")
            print(f"   Time: {segment['start_time']:.1f}s - {segment['end_time']:.1f}s")
            print(f"   Phase: {segment['phase']}")
            if segment['has_transcript']:
                transcript = segment['transcript']
                if len(transcript) > 100:
                    transcript = transcript[:100] + "..."
                print(f"   Transcript: \"{transcript}\"")

    return results

def search_by_phase(surgirag, phase, limit=3):
    """Search segments by surgical phase"""
    phase_segments = [seg for seg in surgirag.data if seg['phase'] == phase]

    if not phase_segments:
        print(f"No segments found for phase: {phase}")
        return []

    # sort chronologically and take first few
    phase_segments.sort(key=lambda x: (x['video_path'], x['start_time']))
    results = phase_segments[:limit]

    print(f"Found {len(results)} segments for phase: {phase}")
    for i, seg in enumerate(results, 1):
        video_name = Path(seg['video_path']).name
        transcript = seg['transcript'][:80] + "..." if len(seg['transcript']) > 80 else seg['transcript']
        print(f"\n{i}. {video_name} ({seg['start_time']:.1f}s-{seg['end_time']:.1f}s)")
        print(f"   Transcript: \"{transcript}\"")

    return results

def search_by_keyword(surgirag, keyword, limit=3):
    """Search segments by keyword in transcript"""
    keyword_segments = []

    for seg in surgirag.data:
        if keyword.lower() in seg['transcript'].lower():
            keyword_segments.append(seg)

    if not keyword_segments:
        print(f"No segments found containing: {keyword}")
        return []

    # Sort by relevance (keyword frequency)
    for seg in keyword_segments:
        seg['keyword_count'] = seg['transcript'].lower().count(keyword.lower())

    keyword_segments.sort(key=lambda x: x['keyword_count'], reverse=True)
    results = keyword_segments[:limit]

    print(f"Found {len(results)} segments containing '{keyword}'")
    for i, seg in enumerate(results, 1):
        video_name = Path(seg['video_path']).name
        transcript = seg['transcript'][:80] + "..." if len(seg['transcript']) > 80 else seg['transcript']
        print(f"\n{i}. {video_name} ({seg['start_time']:.1f}s-{seg['end_time']:.1f}s)")
        print(f"   Count: {seg['keyword_count']} | Transcript: \"{transcript}\"")

    return results

def play_segment_by_index(surgirag, index):
    """Play a specific segment by its index"""
    if index >= len(surgirag.data):
        print(f"Index {index} out of range (max: {len(surgirag.data)-1})")
        return None

    segment = surgirag.data[index]
    print(f"Playing segment {index}:")
    return display_video_segment(segment)

def interactive_search(surgirag):
    """Interactive search interface"""
    print("SurgiRAG Interactive Search")
    print("Commands:")
    print("  ask: <question>        - Ask a question")
    print("  phase: <phase_name>    - Search by phase")
    print("  keyword: <term>        - Search by keyword")
    print("  play: <index>          - Play segment by index")
    print("  quit                   - Exit")
    print()

    while True:
        query = input("Search: ").strip()

        if query.lower() in ['quit', 'exit', 'q']:
            break

        if not query:
            continue

        try:
            if query.startswith('ask:'):
                question = query[4:].strip()
                ask_question(surgirag, question, show_alternatives=False)

            elif query.startswith('phase:'):
                phase = query[6:].strip()
                results = search_by_phase(surgirag, phase, limit=3)

            elif query.startswith('keyword:'):
                keyword = query[8:].strip()
                results = search_by_keyword(surgirag, keyword, limit=3)

            elif query.startswith('play:'):
                index = int(query[5:].strip())
                play_segment_by_index(surgirag, index)

            else:
                # default to question asking
                ask_question(surgirag, query, show_alternatives=False)

        except Exception as e:
            print(f"Error: {e}")

In [9]:
ask_question(surgirag, "Show me trocar insertion")

Question: Show me trocar insertion
Searching for relevant video segments...

Found 2 relevant segments. Showing best match:
Video: Laparoscopic Cholecystectomy： Step- by- Step Surgical guide for Patients and Professionals＂.mp4
Time: 30.0s - 60.0s (30.0s)
Phase: preparation - Patient positioning, trocar insertion, pneumoperitoneum creation
Transcript: "That's the camera port 10mm in the supra umbilical region. This roughly midway between the symphysis pubis and the zephysternum. We create the nommo with the various needle and put a 10mm incision. The incision has to be just adequate. It should not be too big or too small. If it is too big, it will move to and fro. And every time you pull the scope out the trocar also will come out."
Confidence: 0.90

Video clip created: clip_Laparoscopic Cholecystectomy： Step- by- Step Surgical guide for Patients and Professionals＂_1.mp4
File size: 1540217 bytes


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Alternative segments:

2. Laparoscopic Cholecystectomy： Step- by- Step Surgical guide for Patients and Professionals＂.mp4
   Time: 60.0s - 90.0s
   Phase: exploration
   Transcript: "And every time you pull the scope out the trocar also will come out. This has to snugly fit in. Wipe..."


[{'video_id': 'Laparoscopic Cholecystectomy： Step- by- Step Surgical guide for Patients and Professionals＂',
  'video_path': '/content/drive/MyDrive/DATA266/final_project/surgical_videos/Laparoscopic Cholecystectomy： Step- by- Step Surgical guide for Patients and Professionals＂.mp4',
  'segment_idx': 1,
  'start_time': 30,
  'end_time': 60,
  'duration': 30,
  'phase': 'preparation',
  'phase_description': 'Patient positioning, trocar insertion, pneumoperitoneum creation',
  'transcript': "That's the camera port 10mm in the supra umbilical region. This roughly midway between the symphysis pubis and the zephysternum. We create the nommo with the various needle and put a 10mm incision. The incision has to be just adequate. It should not be too big or too small. If it is too big, it will move to and fro. And every time you pull the scope out the trocar also will come out.",
  'has_transcript': True,
  'features': [0.028394468128681183,
   0.03562450408935547,
   0.012536085210740566,
   0